In [ ]:
# In[1]:

import pandas as pd

# Load CSVs into DataFrames and parse timestamp columns as UTC datetimes
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

for df in (metric_df, trace_df, log_df, error_logs_df):
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# --- metric.csv summaries ---
metric_rows = len(metric_df)
metric_min_ts = metric_df["timestamp"].min()
metric_max_ts = metric_df["timestamp"].max()
metric_summary = pd.DataFrame({
    "rows": [metric_rows],
    "min_timestamp_utc": [metric_min_ts.isoformat() if pd.notnull(metric_min_ts) else None],
    "max_timestamp_utc": [metric_max_ts.isoformat() if pd.notnull(metric_max_ts) else None]
})

metric_cmdb_top = metric_df["cmdb_id"].value_counts().head(10).reset_index()
metric_cmdb_top.columns = ["cmdb_id", "count"]

metric_kpi_count = metric_df["kpi_name"].nunique()
metric_kpi_top = metric_df["kpi_name"].value_counts().head(20).reset_index()
metric_kpi_top.columns = ["kpi_name", "count"]

# --- trace.csv summaries ---
trace_rows = len(trace_df)
trace_min_ts = trace_df["timestamp"].min()
trace_max_ts = trace_df["timestamp"].max()
trace_summary = pd.DataFrame({
    "rows": [trace_rows],
    "min_timestamp_utc": [trace_min_ts.isoformat() if pd.notnull(trace_min_ts) else None],
    "max_timestamp_utc": [trace_max_ts.isoformat() if pd.notnull(trace_max_ts) else None]
})

trace_cmdb_top = trace_df["cmdb_id"].value_counts().head(10).reset_index()
trace_cmdb_top.columns = ["cmdb_id", "count"]

trace_name_count = trace_df["trace_name"].nunique()
trace_name_top = trace_df["trace_name"].value_counts().head(20).reset_index()
trace_name_top.columns = ["trace_name", "count"]

# --- log.csv summaries ---
log_rows = len(log_df)
log_min_ts = log_df["timestamp"].min()
log_max_ts = log_df["timestamp"].max()
log_summary = pd.DataFrame({
    "rows": [log_rows],
    "min_timestamp_utc": [log_min_ts.isoformat() if pd.notnull(log_min_ts) else None],
    "max_timestamp_utc": [log_max_ts.isoformat() if pd.notnull(log_max_ts) else None]
})

log_cmdb_top = log_df["cmdb_id"].value_counts().head(10).reset_index()
log_cmdb_top.columns = ["cmdb_id", "count"]

log_name_count = log_df["log_name"].nunique()
log_name_top = log_df["log_name"].value_counts().head(20).reset_index()
log_name_top.columns = ["log_name", "count"]

# --- error_logs.csv summaries ---
error_rows = len(error_logs_df)
error_min_ts = error_logs_df["timestamp"].min()
error_max_ts = error_logs_df["timestamp"].max()
error_distinct_messages = error_logs_df["message"].nunique()
error_summary = pd.DataFrame({
    "rows": [error_rows],
    "min_timestamp_utc": [error_min_ts.isoformat() if pd.notnull(error_min_ts) else None],
    "max_timestamp_utc": [error_max_ts.isoformat() if pd.notnull(error_max_ts) else None],
    "distinct_message_count": [error_distinct_messages]
})

# Show up to first 20 rows as samples (timestamp, cmdb_id, message)
error_samples = error_logs_df.loc[:, ["timestamp", "cmdb_id", "message"]].head(20).copy()
# convert each Timestamp to ISO string
error_samples["timestamp"] = error_samples["timestamp"].apply(lambda x: x.isoformat() if pd.notnull(x) else None)

# Final compact outputs (do not show full raw DataFrames)
metric_summary, metric_cmdb_top, pd.DataFrame({"distinct_kpi_count":[metric_kpi_count]}), metric_kpi_top, \
trace_summary, trace_cmdb_top, pd.DataFrame({"distinct_trace_count":[trace_name_count]}), trace_name_top, \
log_summary, log_cmdb_top, pd.DataFrame({"distinct_log_count":[log_name_count]}), log_name_top, \
error_summary, error_samples

```
Out[1]:
```
summary = (
    "Summary of telemetry load (UTC):\n"
    "- metric.csv: 1775 rows from 2024-01-16T09:32:00+00:00 to 2024-01-16T09:56:00+00:00. "
    "Top cmdb_ids: adservice (200), currencyservice (175), recommendationservice (175), emailservice (175), checkoutservice (150). "
    "There are 8 distinct KPIs; most frequent KPI names: cpu, workload, socket, mem (each 275), followed by latency-50 and latency-90.\n"
    "- trace.csv: 3248 rows over the same time range. "
    "Top cmdb_ids: checkoutservice (692), root (632), frontendservice (572), recommendationservice (424), productcatalogservice (340). "
    "There are 44 distinct trace names; the most common trace groups are trace.from_root.* and traces from checkout/frontend/productcatalog services.\n"
    "- log.csv: 366 rows over the same time range. "
    "Top cmdb_ids are evenly present (adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice ~40 each). "
    "There are 2 distinct log names: log.error_count and log.row_count (each 183 occurrences).\n"
    "- error_logs.csv: 0 rows (no error log messages present).\n\n"
    "Overall: metric and trace telemetry are plentiful and show activity concentrated in adservice and checkout/frontend-related services; logs are available but limited to two aggregated log metrics; there are no raw error log messages in error_logs.csv."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   rows          min_timestamp_utc          max_timestamp_utc
0  1775  2024-01-16T09:32:00+00:00  2024-01-16T09:56:00+00:00,                  cmdb_id  count
0              adservice    200
1        currencyservice    175
2  recommendationservice    175
3           emailservice    175
4        checkoutservice    150
5            cartservice    150
6         paymentservice    150
7               frontend    150
8        shippingservice    150
9  productcatalogservice    150,    distinct_kpi_count
0                   8,      kpi_name  count
0         cpu    275
1    workload    275
2      socket    275
3         mem    275
4  latency-50    250
5  latency-90    250
6      diskio    100
7       error     75,    rows          min_timestamp_utc          max_timestamp_utc
0  3248  2024-01-16T09:32:00+00:00  2024-01-16T09:56:00+00:00,                  cmdb_id  count
0        checkoutservice    692
1                   root    632
2        frontendservice    572
3  recommendationservice    424
4  productcatalogservice    340
5        currencyservice    260
6         paymentservice    168
7           emailservice    160,    distinct_trace_count
0                    44,                                       trace_name  count
0                  trace.from_root.duration_mean    158
1                   trace.from_root.duration_p95    158
2                     trace.from_root.error_rate    158
3                      trace.from_root.row_count    158
4           trace.from_checkoutservice.row_count    106
5          trace.from_checkoutservice.error_rate    106
6        trace.from_checkoutservice.duration_p95    106
7       trace.from_checkoutservice.duration_mean    106
8           trace.from_frontendservice.row_count    100
9          trace.from_frontendservice.error_rate    100
10       trace.from_frontendservice.duration_p95    100
11      trace.from_frontendservice.duration_mean    100
12      trace.to_productcatalogservice.row_count     85
13     trace.to_productcatalogservice.error_rate     85
14   trace.to_productcatalogservice.duration_p95     85
15  trace.to_productcatalogservice.duration_mean     85
16        trace.to_checkoutservice.duration_mean     67
17         trace.to_checkoutservice.duration_p95     67
18           trace.to_checkoutservice.error_rate     67
19            trace.to_checkoutservice.row_count     67,    rows          min_timestamp_utc          max_timestamp_utc
0   366  2024-01-16T09:32:00+00:00  2024-01-16T09:56:00+00:00,                  cmdb_id  count
0              adservice     40
1            cartservice     40
2        checkoutservice     40
3        currencyservice     40
4           emailservice     40
5               frontend     40
6         paymentservice     40
7  recommendationservice     40
8        shippingservice     40
9                  redis      6,    distinct_log_count
0                   2,           log_name  count
0  log.error_count    183
1    log.row_count    183,    rows min_timestamp_utc max_timestamp_utc  distinct_message_count
0     0              None              None                       0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse existing metric_df from kernel
components = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# 1) restrict to components for threshold calculation
mask = metric_df["cmdb_id"].isin(components)
metric_comp = metric_df[mask].copy()

# 2) compute global P95, P15, P05 per component-kpi using full series (not windowed)
thresholds = (
    metric_comp
    .groupby(["cmdb_id", "kpi_name"])["value"]
    .agg(P95=lambda x: x.quantile(0.95), P15=lambda x: x.quantile(0.15), P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 3) filter metric rows to the incident window
start = pd.to_datetime("2024-01-16T09:29:02+00:00", utc=True)
end   = pd.to_datetime("2024-01-16T09:59:02+00:00", utc=True)
window_df = metric_comp[(metric_comp["timestamp"] >= start) & (metric_comp["timestamp"] <= end)].copy()

# 4) merge thresholds into windowed rows so each row has its global thresholds
window_with_thresh = window_df.merge(thresholds, on=["cmdb_id", "kpi_name"], how="left")

# flags for anomalies
window_with_thresh["is_high"] = window_with_thresh["value"] >= window_with_thresh["P95"]
window_with_thresh["is_low"] = window_with_thresh["value"] <= window_with_thresh["P05"]

# group aggregations within the window
grp = window_with_thresh.groupby(["cmdb_id", "kpi_name"])

window_points = grp["value"].size().rename("window_points")
high_count = grp["is_high"].sum().rename("high_count")
low_count = grp["is_low"].sum().rename("low_count")
max_value = grp["value"].max().rename("max_value")
min_value = grp["value"].min().rename("min_value")

earliest_high = (
    window_with_thresh[window_with_thresh["is_high"]]
    .groupby(["cmdb_id", "kpi_name"])["timestamp"]
    .min()
    .rename("earliest_high_ts")
)
earliest_low = (
    window_with_thresh[window_with_thresh["is_low"]]
    .groupby(["cmdb_id", "kpi_name"])["timestamp"]
    .min()
    .rename("earliest_low_ts")
)

# assemble aggregated table
agg = pd.concat([window_points, high_count, low_count, max_value, min_value], axis=1).reset_index()

# attach earliest timestamps (may be missing for many groups)
agg = agg.merge(earliest_high.reset_index(), on=["cmdb_id", "kpi_name"], how="left")
agg = agg.merge(earliest_low.reset_index(), on=["cmdb_id", "kpi_name"], how="left")

# attach thresholds
agg = agg.merge(thresholds, on=["cmdb_id", "kpi_name"], how="left")

# compute ratios (with guards for zero/NaN)
agg["max_over_p95"] = np.where(agg["P95"] > 0, agg["max_value"] / agg["P95"], np.nan)
agg["p05_over_min"] = np.where(agg["min_value"] > 0, agg["P05"] / agg["min_value"], np.nan)

# format earliest timestamps as ISO strings (or None)
agg["earliest_high_ts"] = agg["earliest_high_ts"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
agg["earliest_low_ts"] = agg["earliest_low_ts"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")

# replace NaT formatting results with None
agg["earliest_high_ts"] = agg["earliest_high_ts"].replace({pd.NaT: None, "NaT": None})
agg["earliest_low_ts"] = agg["earliest_low_ts"].replace({pd.NaT: None, "NaT": None})

# round ratio columns for compactness
agg["max_over_p95"] = agg["max_over_p95"].round(3)
agg["p05_over_min"] = agg["p05_over_min"].round(3)

# 5) filter to entries with at least one anomaly and sort by combined anomaly count descending
agg["combined_anomalies"] = agg["high_count"] + agg["low_count"]
result = (
    agg[agg["combined_anomalies"] > 0]
    .sort_values(by="combined_anomalies", ascending=False)
    .loc[:, ["cmdb_id", "kpi_name", "P95", "P15", "P05", "window_points",
             "high_count", "low_count", "earliest_high_ts", "earliest_low_ts",
             "max_value", "max_over_p95", "min_value", "p05_over_min"]]
    .head(20)
    .reset_index(drop=True)
)

# Keep full intermediate dataframes in kernel variables for later use
thresholds_df = thresholds
window_rows_df = window_df
window_with_thresholds_df = window_with_thresh
aggregated_df = agg

# display compact result (top 20)
result

```
Out[2]:
```
Summary of findings (plain English):

- Scope and method: I analyzed metrics for checkoutservice, currencyservice, emailservice, productcatalogservice, and recommendationservice. Global P95/P15/P05 thresholds were computed from each component-kpi's full time series, then I inspected metric points inside the incident window 2024-01-16T09:29:02+00:00 to 2024-01-16T09:59:02+00:00.

- Overall result: Multiple component–KPI pairs show anomalies inside the window. Many anomalies are concentrated at 2024-01-16T09:32:00+00:00 (earliest anomaly timestamp for many entries).

- Top affected component–KPIs (sorted by combined anomaly count):
  1. checkoutservice — socket: all 25 points equal to 9.0 (P95=P05=9.0) so every point counts as both high and low (25 high, 25 low), earliest at 2024-01-16T09:32:00Z.
  2. recommendationservice — diskio: all zeros (P95=P05=0.0), 25 high and 25 low by equality; earliest 2024-01-16T09:32:00Z.
  3. currencyservice — error: all zeros (P95=P05=0.0), 25 high and 25 low; earliest 2024-01-16T09:32:00Z.
  4. emailservice — diskio: all zeros (P95=P05=0.0), 25 high and 25 low; earliest 2024-01-16T09:32:00Z.
  5. currencyservice — socket; productcatalogservice — socket; emailservice — socket: similarly show all-or-most points equal to the threshold (25 high, ~24–25 low), earliest anomalies at 09:32:00Z. Many socket metrics have max/min equal to P95/P05 (ratios ~1.0).
  6. Several KPIs show smaller counts of anomalies (high_count and low_count = 2 each) for checkoutservice (cpu, workload, mem, latency-50, latency-90), currencyservice (cpu, mem, latency-50, latency-90), emailservice (cpu, latency-50, latency-90), and recommendationservice (socket also had mixed high/low). For these, max_over_p95 is modestly >1 (examples: checkoutservice cpu ~1.134, checkoutservice latency-90 ~1.697) and p05_over_min values indicate the min dipped below P05 in some cases.

- Important note about interpretation: many entries have P95 == P05 (or values exactly equal to thresholds). Because anomalies were defined with >=P95 and <=P05, points exactly equal to these thresholds are counted as both high and low. That explains the many rows showing equal high_count and low_count (25 each). These cases often reflect constant or all-zero series rather than classic “spikes”.

- Practical takeaways:
  - Investigate socket-related metrics across multiple services (checkoutservice, currencyservice, productcatalogservice, emailservice, recommendationservice) — they show persistent values at threshold and could indicate sustained socket conditions or normalization artifacts.
  - Disk I/O and error metrics for some services are constant zeros during the window; confirm whether that is expected (no activity) or indicates ingestion/normalization issues.
  - Checkoutservice shows small but real increases in CPU, workload, memory and latency (max_over_p95 > 1), with earliest high anomalies at ~09:32–09:36Z — worth inspecting traces/logs for checkoutservice around that time.
  - Confirm anomaly definition (>=P95 and <=P05) if you want to avoid counting equality cases as both high and low; adjust thresholds or inequality rules if necessary.

If you want, I can (next) (a) show the exact top 20 rows again in tabular form, (b) re-calculate anomalies using strict >P95 and <P05 to avoid equality double-counting, or (c) examine traces/logs for checkoutservice around 09:32–09:36Z. Which would you prefer?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name           P95           P15           P05  window_points  high_count  low_count          earliest_high_ts           earliest_low_ts     max_value  max_over_p95     min_value  p05_over_min
0         checkoutservice      socket  9.000000e+00  9.000000e+00  9.000000e+00             25          25         25  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000  9.000000e+00         1.000  9.000000e+00         1.000
1   recommendationservice      diskio  0.000000e+00  0.000000e+00  0.000000e+00             25          25         25  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000  0.000000e+00           NaN  0.000000e+00           NaN
2         currencyservice       error  0.000000e+00  0.000000e+00  0.000000e+00             25          25         25  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000  0.000000e+00           NaN  0.000000e+00           NaN
3            emailservice      diskio  0.000000e+00  0.000000e+00  0.000000e+00             25          25         25  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000  0.000000e+00           NaN  0.000000e+00           NaN
4         currencyservice      socket  3.000000e+00  3.000000e+00  3.000000e+00             25          25         25  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000  3.000000e+00         1.000  3.000000e+00         1.000
5   productcatalogservice      socket  4.000000e+00  4.000000e+00  4.000000e+00             25          25         25  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000  4.000000e+00         1.000  4.000000e+00         1.000
6            emailservice      socket  3.000000e+00  3.000000e+00  3.000000e+00             25          25         24  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000  3.200000e+00         1.067  3.000000e+00         1.000
7   recommendationservice      socket  2.618000e+01  4.000000e+00  4.000000e+00             25           2         12  2024-01-16T09:48:00+0000  2024-01-16T09:32:00+0000  2.630000e+01         1.005  4.000000e+00         1.000
8            emailservice  latency-90  4.739118e-03  4.600000e-03  4.600000e-03             25           2         10  2024-01-16T09:36:00+0000  2024-01-16T09:33:00+0000  4.797133e-03         1.012  4.600000e-03         1.000
9            emailservice  latency-50  3.077288e-03  3.000000e-03  3.000000e-03             25           2         10  2024-01-16T09:36:00+0000  2024-01-16T09:33:00+0000  3.109518e-03         1.010  3.000000e-03         1.000
10        checkoutservice         cpu  4.538865e-01  3.313383e-01  2.938164e-01             25           2          2  2024-01-16T09:32:00+0000  2024-01-16T09:37:00+0000  5.147674e-01         1.134  2.616598e-01         1.123
11        checkoutservice    workload  1.080780e+00  7.123967e-01  5.544567e-01             25           2          2  2024-01-16T09:35:00+0000  2024-01-16T09:54:00+0000  1.127983e+00         1.044  5.330000e-01         1.040
12        checkoutservice         mem  1.197586e+07  1.142637e+07  1.137592e+07             25           2          2  2024-01-16T09:36:00+0000  2024-01-16T09:32:00+0000  1.203555e+07         1.005  1.122543e+07         1.013
13        currencyservice  latency-50  3.997674e-03  3.720995e-03  3.692106e-03             25           2          2  2024-01-16T09:32:00+0000  2024-01-16T09:50:00+0000  4.008564e-03         1.003  3.644642e-03         1.013
14        currencyservice         cpu  2.972365e+01  2.655896e+01  2.631345e+01             25           2          2  2024-01-16T09:32:00+0000  2024-01-16T09:35:00+0000  3.008827e+01         1.012  2.587013e+01         1.017
15        checkoutservice  latency-90  3.394511e-01  2.271245e-01  2.268213e-01             25           2          2  2024-01-16T09:36:00+0000  2024-01-16T09:33:00+0000  5.760562e-01         1.697  2.187018e-01         1.037
16        checkoutservice  latency-50  1.478744e-01  1.128819e-01  9.775921e-02             25           2          2  2024-01-16T09:36:00+0000  2024-01-16T09:47:00+0000  1.535746e-01         1.039  7.671084e-02         1.274
17           emailservice         cpu  3.337029e-01  2.892492e-01  2.858781e-01             25           2          2  2024-01-16T09:32:00+0000  2024-01-16T09:37:00+0000  3.366260e-01         1.009  2.855782e-01         1.001
18        currencyservice  latency-90  6.595512e-02  4.275190e-02  4.085006e-02             25           2          2  2024-01-16T09:33:00+0000  2024-01-16T09:51:00+0000  6.692314e-02         1.015  3.970643e-02         1.029
19        currencyservice         mem  4.547830e+07  4.486865e+07  4.473012e+07             25           2          2  2024-01-16T09:40:00+0000  2024-01-16T09:44:00+0000  4.598989e+07         1.011  4.469514e+07         1.001```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# reuse/define components and trace_df from kernel
components = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# 1) restrict trace rows to selected components for threshold calculation
trace_comp = trace_df[trace_df["cmdb_id"].isin(components)].copy()

# 1) compute global P95, P15, P05 per component-trace_name using full series
thresholds_traces_df = (
    trace_comp
    .groupby(["cmdb_id", "trace_name"])["value"]
    .agg(P95=lambda x: x.quantile(0.95), P15=lambda x: x.quantile(0.15), P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) filter trace rows to the incident window
start = pd.to_datetime("2024-01-16T09:29:02+00:00", utc=True)
end   = pd.to_datetime("2024-01-16T09:59:02+00:00", utc=True)
window_traces_df = trace_comp[(trace_comp["timestamp"] >= start) & (trace_comp["timestamp"] <= end)].copy()

# 3) merge thresholds into windowed rows
window_with_thresholds_traces_df = window_traces_df.merge(thresholds_traces_df, on=["cmdb_id", "trace_name"], how="left")

# anomaly flags
window_with_thresholds_traces_df["is_high"] = window_with_thresholds_traces_df["value"] >= window_with_thresholds_traces_df["P95"]
window_with_thresholds_traces_df["is_low"] = window_with_thresholds_traces_df["value"] <= window_with_thresholds_traces_df["P05"]

# group aggregations within the window
grp = window_with_thresholds_traces_df.groupby(["cmdb_id", "trace_name"])
window_points = grp["value"].size().rename("window_points")
high_count = grp["is_high"].sum().rename("high_count")
low_count = grp["is_low"].sum().rename("low_count")
max_value = grp["value"].max().rename("max_value")
min_value = grp["value"].min().rename("min_value")

earliest_high = (
    window_with_thresholds_traces_df[window_with_thresholds_traces_df["is_high"]]
    .groupby(["cmdb_id", "trace_name"])["timestamp"]
    .min()
    .rename("earliest_high_ts")
)
earliest_low = (
    window_with_thresholds_traces_df[window_with_thresholds_traces_df["is_low"]]
    .groupby(["cmdb_id", "trace_name"])["timestamp"]
    .min()
    .rename("earliest_low_ts")
)

# assemble aggregated table
aggregated_traces_df = pd.concat([window_points, high_count, low_count, max_value, min_value], axis=1).reset_index()
aggregated_traces_df = aggregated_traces_df.merge(earliest_high.reset_index(), on=["cmdb_id", "trace_name"], how="left")
aggregated_traces_df = aggregated_traces_df.merge(earliest_low.reset_index(), on=["cmdb_id", "trace_name"], how="left")
aggregated_traces_df = aggregated_traces_df.merge(thresholds_traces_df, on=["cmdb_id", "trace_name"], how="left")

# compute ratios
aggregated_traces_df["max_over_p95"] = np.where(aggregated_traces_df["P95"] > 0, aggregated_traces_df["max_value"] / aggregated_traces_df["P95"], np.nan)
aggregated_traces_df["p05_over_min"] = np.where(aggregated_traces_df["min_value"] > 0, aggregated_traces_df["P05"] / aggregated_traces_df["min_value"], np.nan)

# format earliest timestamps as ISO strings
aggregated_traces_df["earliest_high_ts"] = aggregated_traces_df["earliest_high_ts"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
aggregated_traces_df["earliest_low_ts"]  = aggregated_traces_df["earliest_low_ts"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
aggregated_traces_df["earliest_high_ts"] = aggregated_traces_df["earliest_high_ts"].replace({pd.NaT: None, "NaT": None})
aggregated_traces_df["earliest_low_ts"]  = aggregated_traces_df["earliest_low_ts"].replace({pd.NaT: None, "NaT": None})

# round ratio columns
aggregated_traces_df["max_over_p95"] = aggregated_traces_df["max_over_p95"].round(3)
aggregated_traces_df["p05_over_min"] = aggregated_traces_df["p05_over_min"].round(3)

# 4) filter to entries with at least one anomaly and sort by combined anomaly count descending
aggregated_traces_df["combined_anomalies"] = aggregated_traces_df["high_count"] + aggregated_traces_df["low_count"]
result_traces = (
    aggregated_traces_df[aggregated_traces_df["combined_anomalies"] > 0]
    .sort_values(by="combined_anomalies", ascending=False)
    .loc[:, ["cmdb_id", "trace_name", "P95", "P15", "P05", "window_points",
             "high_count", "low_count", "earliest_high_ts", "earliest_low_ts",
             "max_value", "max_over_p95", "min_value", "p05_over_min"]]
    .head(20)
    .reset_index(drop=True)
)

# 5) summary: top 5 trace_names across components by total anomaly counts and indicate direction ('to_' or 'from_')
# aggregate combined anomalies by trace_name (across components)
trace_anom_by_name = (
    aggregated_traces_df.groupby("trace_name")["combined_anomalies"]
    .sum()
    .reset_index()
    .sort_values(by="combined_anomalies", ascending=False)
)
trace_anom_by_name["direction"] = trace_anom_by_name["trace_name"].apply(
    lambda s: ("to" if ".to_" in s or "to_" in s else ("from" if ".from_" in s or "from_" in s else "unknown"))
)
top5_trace_summary = trace_anom_by_name.head(5).reset_index(drop=True)

# Keep intermediate dataframes in kernel for later use:
# thresholds_traces_df, window_traces_df, window_with_thresholds_traces_df, aggregated_traces_df

# Display compact outputs: result_traces (top 20) and top5_trace_summary (top 5)
result_traces, top5_trace_summary

```
Out[3]:
```
Summary (plain English):

- Across checkoutservice, currencyservice, emailservice, productcatalogservice, and recommendationservice, trace error_rate metrics during the incident window are uniformly zero and have global P95/P05 = 0, so every point equals the threshold and was counted as both "high" and "low". Earliest such points are at 2024-01-16T09:32:00Z.  
- This pattern produces many anomaly counts but likely reflects constant zero values (either expected “no errors” or a telemetry/normalization artifact), not classic spikes.  
- Top 5 trace names by total anomaly count (direction):  
  1) trace.from_root.error_rate — from  
  2) trace.from_checkoutservice.error_rate — from  
  3) trace.from_frontendservice.error_rate — from  
  4) trace.from_recommendationservice.error_rate — from  
  5) trace.to_productcatalogservice.error_rate — to  
- Interpretation: anomalies are concentrated in "from_*" error_rate traces (propagating from frontend/checkout/recommendation) and a notable "to_productcatalogservice" signal — indicates propagation paths to inspect, but the zero-valued nature suggests first verifying telemetry correctness.  
- Recommendation: confirm whether error_rate = 0 is expected; if not, check trace ingestion/normalization around 09:32Z and inspect traces for the listed "from_*" → "to_*" paths (especially calls toward productcatalogservice).

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  P95  P15  P05  window_points  high_count  low_count          earliest_high_ts           earliest_low_ts  max_value  max_over_p95  min_value  p05_over_min
0         checkoutservice                   trace.from_root.error_rate  0.0  0.0  0.0             24          24         24  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
1         currencyservice                   trace.from_root.error_rate  0.0  0.0  0.0             24          24         24  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
2   productcatalogservice                   trace.from_root.error_rate  0.0  0.0  0.0             24          24         24  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
3         checkoutservice        trace.from_checkoutservice.error_rate  0.0  0.0  0.0             23          23         23  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
4         checkoutservice          trace.to_checkoutservice.error_rate  0.0  0.0  0.0             23          23         23  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
5   recommendationservice    trace.to_recommendationservice.error_rate  0.0  0.0  0.0             22          22         22  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
6   recommendationservice                   trace.from_root.error_rate  0.0  0.0  0.0             22          22         22  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
7   recommendationservice  trace.from_recommendationservice.error_rate  0.0  0.0  0.0             22          22         22  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
8         checkoutservice    trace.to_productcatalogservice.error_rate  0.0  0.0  0.0             21          21         21  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
9         currencyservice        trace.from_checkoutservice.error_rate  0.0  0.0  0.0             21          21         21  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
10        checkoutservice           trace.to_paymentservice.error_rate  0.0  0.0  0.0             21          21         21  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
11        checkoutservice          trace.to_currencyservice.error_rate  0.0  0.0  0.0             21          21         21  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
12  productcatalogservice        trace.from_checkoutservice.error_rate  0.0  0.0  0.0             21          21         21  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
13  recommendationservice    trace.to_productcatalogservice.error_rate  0.0  0.0  0.0             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
14  recommendationservice        trace.from_frontendservice.error_rate  0.0  0.0  0.0             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
15  productcatalogservice        trace.from_frontendservice.error_rate  0.0  0.0  0.0             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
16        currencyservice        trace.from_frontendservice.error_rate  0.0  0.0  0.0             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
17           emailservice        trace.from_checkoutservice.error_rate  0.0  0.0  0.0             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
18           emailservice                   trace.from_root.error_rate  0.0  0.0  0.0             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN
19  productcatalogservice  trace.from_recommendationservice.error_rate  0.0  0.0  0.0             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000        0.0           NaN        0.0           NaN,                                     trace_name  combined_anomalies direction
0                   trace.from_root.error_rate                 228      from
1        trace.from_checkoutservice.error_rate                 170      from
2        trace.from_frontendservice.error_rate                 160      from
3  trace.from_recommendationservice.error_rate                  84      from
4    trace.to_productcatalogservice.error_rate                  82        to)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# reuse trace_df and components from kernel state
components = ["checkoutservice", "currencyservice", "emailservice", "productcatalogservice", "recommendationservice"]

# 1) identify trace_name values containing 'duration_mean' or 'duration_p95' for selected components
trace_comp = trace_df[trace_df["cmdb_id"].isin(components)].copy()
mask_latency_names = trace_comp["trace_name"].str.contains("duration_mean|duration_p95", case=False, na=False)
trace_latency_df = trace_comp[mask_latency_names].copy()

# store the set of relevant trace_names
latency_trace_names = trace_latency_df["trace_name"].unique()

# 2) compute global P95, P15, P05 per component-trace_name using full series (global thresholds)
thresholds_traces_latency_df = (
    trace_latency_df
    .groupby(["cmdb_id", "trace_name"])["value"]
    .agg(P95=lambda x: x.quantile(0.95), P15=lambda x: x.quantile(0.15), P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 3) filter trace rows to the incident window
start = pd.to_datetime("2024-01-16T09:29:02+00:00", utc=True)
end   = pd.to_datetime("2024-01-16T09:59:02+00:00", utc=True)
window_traces_latency_df = trace_latency_df[(trace_latency_df["timestamp"] >= start) & (trace_latency_df["timestamp"] <= end)].copy()

# 4) merge thresholds into windowed rows and compute anomaly metrics
window_with_thresholds_traces_latency_df = window_traces_latency_df.merge(thresholds_traces_latency_df, on=["cmdb_id", "trace_name"], how="left")
window_with_thresholds_traces_latency_df["is_high"] = window_with_thresholds_traces_latency_df["value"] >= window_with_thresholds_traces_latency_df["P95"]

grp = window_with_thresholds_traces_latency_df.groupby(["cmdb_id", "trace_name"])
window_points = grp["value"].size().rename("window_points")
high_count = grp["is_high"].sum().rename("high_count")
earliest_high = (
    window_with_thresholds_traces_latency_df[window_with_thresholds_traces_latency_df["is_high"]]
    .groupby(["cmdb_id", "trace_name"])["timestamp"]
    .min()
    .rename("earliest_high_ts")
)
max_value = grp["value"].max().rename("max_value")

aggregated_traces_latency_df = pd.concat([window_points, high_count, max_value], axis=1).reset_index()
aggregated_traces_latency_df = aggregated_traces_latency_df.merge(earliest_high.reset_index(), on=["cmdb_id", "trace_name"], how="left")
aggregated_traces_latency_df = aggregated_traces_latency_df.merge(thresholds_traces_latency_df, on=["cmdb_id", "trace_name"], how="left")

# compute max_over_p95 with guard and round
aggregated_traces_latency_df["max_over_p95"] = np.where(aggregated_traces_latency_df["P95"] > 0,
                                                       (aggregated_traces_latency_df["max_value"] / aggregated_traces_latency_df["P95"]).round(3),
                                                       np.nan)

# format earliest_high_ts as ISO strings (or None)
aggregated_traces_latency_df["earliest_high_ts"] = aggregated_traces_latency_df["earliest_high_ts"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
aggregated_traces_latency_df["earliest_high_ts"] = aggregated_traces_latency_df["earliest_high_ts"].replace({pd.NaT: None, "NaT": None})

# exclude entries with window_points == 0
aggregated_traces_latency_df = aggregated_traces_latency_df[aggregated_traces_latency_df["window_points"] > 0].copy()

# 5) build compact table of entries with high_count > 0, sorted by high_count desc, limit top 20
result_latency = (
    aggregated_traces_latency_df[aggregated_traces_latency_df["high_count"] > 0]
    .sort_values(by="high_count", ascending=False)
    .loc[:, ["cmdb_id", "trace_name", "P95", "P15", "P05", "window_points", "high_count", "earliest_high_ts", "max_value", "max_over_p95"]]
    .head(20)
    .reset_index(drop=True)
)

# 6) three-line summary: top 3 trace entries by largest max_over_p95 (include direction 'to'/'from')
# consider only rows with non-null max_over_p95
candidates_for_top3 = result_latency.dropna(subset=["max_over_p95"]).copy()
top3_by_ratio = candidates_for_top3.sort_values(by="max_over_p95", ascending=False).head(3).reset_index(drop=True)

def direction_of(name):
    if "to_" in name:
        return "to"
    if "from_" in name:
        return "from"
    return "unknown"

top3_summary_rows = []
for _, r in top3_by_ratio.iterrows():
    entry = f"{r['cmdb_id']} + {r['trace_name']}"
    dirn = direction_of(r["trace_name"])
    top3_summary_rows.append({"entry": entry, "direction": dirn, "max_over_p95": r["max_over_p95"]})

top3_summary = pd.DataFrame(top3_summary_rows).head(3)

# Keep intermediate dataframes in kernel variables for later steps
# thresholds_traces_latency_df, window_traces_latency_df, window_with_thresholds_traces_latency_df, aggregated_traces_latency_df

# Display compact outputs: result_latency (top 20) and top3_summary (3 lines)
result_latency, top3_summary

```
Out[4]:
```
Summary (plain English):

- Many latency traces had a small number of high points (typically 2) during the incident window; most window series have ~21–24 points. The largest latency spikes are concentrated on traces involving currencyservice and checkoutservice calls.
- Top 3 trace latency hotspots:
  1) currencyservice + trace.from_root.duration_p95 — direction: from — max_over_p95 ≈ 22.8 (earliest high ~2024-01-16T09:40:00Z).  
  2) currencyservice + trace.from_root.duration_mean — direction: from — max_over_p95 ≈ 6.57 (earliest high ~2024-01-16T09:52:00Z).  
  3) checkoutservice + trace.to_paymentservice.duration_p95 — direction: to — max_over_p95 ≈ 1.92 (earliest high ~2024-01-16T09:45:00Z).
- Actionable next steps: investigate currencyservice inbound latency around ~09:40–09:52Z (calls originating "from_root"), and inspect checkoutservice→paymentservice calls around ~09:45Z to identify downstream latency propagation.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                      trace_name       P95       P15       P05  window_points  high_count          earliest_high_ts  max_value  max_over_p95
0         checkoutservice        trace.from_checkoutservice.duration_mean  0.013098  0.003571  0.003379             23           2  2024-01-16T09:35:00+0000   0.020922         1.597
1         checkoutservice         trace.from_checkoutservice.duration_p95  0.006350  0.005583  0.004884             23           2  2024-01-16T09:36:00+0000   0.006606         1.040
2         checkoutservice                    trace.from_root.duration_p95  0.109761  0.074094  0.054610             24           2  2024-01-16T09:50:00+0000   0.117598         1.071
3         checkoutservice                   trace.from_root.duration_mean  0.028371  0.015552  0.012472             24           2  2024-01-16T09:50:00+0000   0.032977         1.162
4         checkoutservice          trace.to_currencyservice.duration_mean  0.001994  0.000125  0.000116             21           2  2024-01-16T09:35:00+0000   0.002427         1.217
5         checkoutservice           trace.to_currencyservice.duration_p95  0.000215  0.000172  0.000149             21           2  2024-01-16T09:37:00+0000   0.000219         1.016
6         checkoutservice          trace.to_checkoutservice.duration_mean  0.013098  0.003571  0.003379             23           2  2024-01-16T09:35:00+0000   0.020922         1.597
7         checkoutservice           trace.to_checkoutservice.duration_p95  0.006350  0.005583  0.004884             23           2  2024-01-16T09:36:00+0000   0.006606         1.040
8         checkoutservice     trace.to_productcatalogservice.duration_p95  0.000033  0.000021  0.000020             21           2  2024-01-16T09:33:00+0000   0.000041         1.261
9         checkoutservice    trace.to_productcatalogservice.duration_mean  0.000021  0.000015  0.000015             21           2  2024-01-16T09:41:00+0000   0.000023         1.115
10        checkoutservice            trace.to_paymentservice.duration_p95  0.001275  0.000290  0.000265             21           2  2024-01-16T09:45:00+0000   0.002449         1.921
11        checkoutservice           trace.to_paymentservice.duration_mean  0.000446  0.000237  0.000232             21           2  2024-01-16T09:45:00+0000   0.000642         1.440
12  recommendationservice   trace.from_recommendationservice.duration_p95  0.199980  0.004497  0.004435             22           2  2024-01-16T09:44:00+0000   0.201090         1.006
13  recommendationservice  trace.from_recommendationservice.duration_mean  0.101443  0.003299  0.003268             22           2  2024-01-16T09:45:00+0000   0.116607         1.149
14  recommendationservice                   trace.from_root.duration_mean  0.087419  0.000047  0.000043             22           2  2024-01-16T09:50:00+0000   0.098908         1.131
15  recommendationservice                    trace.from_root.duration_p95  0.163907  0.000067  0.000063             22           2  2024-01-16T09:49:00+0000   0.197467         1.205
16        currencyservice        trace.from_checkoutservice.duration_mean  0.001994  0.000125  0.000116             21           2  2024-01-16T09:35:00+0000   0.002427         1.217
17        currencyservice         trace.from_checkoutservice.duration_p95  0.000215  0.000172  0.000149             21           2  2024-01-16T09:37:00+0000   0.000219         1.016
18        currencyservice                    trace.from_root.duration_p95  0.002423  0.000062  0.000057             24           2  2024-01-16T09:40:00+0000   0.055212        22.785
19        currencyservice                   trace.from_root.duration_mean  0.000904  0.000049  0.000048             24           2  2024-01-16T09:52:00+0000   0.005936         6.571,                                                entry direction  max_over_p95
0     currencyservice + trace.from_root.duration_p95      from        22.785
1    currencyservice + trace.from_root.duration_mean      from         6.571
2  checkoutservice + trace.to_paymentservice.dura...        to         1.921)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# reuse existing log_df from kernel
target_cmdbs = ["checkoutservice", "currencyservice"]

# 1) compute global P95 and P05 per cmdb_id-log_name using the full series (for the two cmdb_ids)
logs_for_thresholds = log_df[log_df["cmdb_id"].isin(target_cmdbs)].copy()
thresholds_logs_df = (
    logs_for_thresholds
    .groupby(["cmdb_id", "log_name"])["value"]
    .agg(P95=lambda x: x.quantile(0.95), P05=lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) filter log rows to the incident window
start = pd.to_datetime("2024-01-16T09:29:02+00:00", utc=True)
end   = pd.to_datetime("2024-01-16T09:59:02+00:00", utc=True)
window_logs_df = logs_for_thresholds[(logs_for_thresholds["timestamp"] >= start) & (logs_for_thresholds["timestamp"] <= end)].copy()

# 3) merge thresholds into windowed rows and compute anomaly flags
window_with_thresholds_logs_df = window_logs_df.merge(thresholds_logs_df, on=["cmdb_id", "log_name"], how="left")
window_with_thresholds_logs_df["is_high"] = window_with_thresholds_logs_df["value"] >= window_with_thresholds_logs_df["P95"]
window_with_thresholds_logs_df["is_low"] = window_with_thresholds_logs_df["value"] <= window_with_thresholds_logs_df["P05"]

# group aggregations within the window
grp = window_with_thresholds_logs_df.groupby(["cmdb_id", "log_name"])
window_points = grp["value"].size().rename("window_points")
high_count = grp["is_high"].sum().rename("high_count")
low_count = grp["is_low"].sum().rename("low_count")
max_value = grp["value"].max().rename("max_value")
min_value = grp["value"].min().rename("min_value")

earliest_high = (
    window_with_thresholds_logs_df[window_with_thresholds_logs_df["is_high"]]
    .groupby(["cmdb_id", "log_name"])["timestamp"]
    .min()
    .rename("earliest_high_ts")
)
earliest_low = (
    window_with_thresholds_logs_df[window_with_thresholds_logs_df["is_low"]]
    .groupby(["cmdb_id", "log_name"])["timestamp"]
    .min()
    .rename("earliest_low_ts")
)

# assemble aggregated table
aggregated_logs_df = pd.concat([window_points, high_count, low_count, max_value, min_value], axis=1).reset_index()
aggregated_logs_df = aggregated_logs_df.merge(earliest_high.reset_index(), on=["cmdb_id", "log_name"], how="left")
aggregated_logs_df = aggregated_logs_df.merge(earliest_low.reset_index(), on=["cmdb_id", "log_name"], how="left")
aggregated_logs_df = aggregated_logs_df.merge(thresholds_logs_df, on=["cmdb_id", "log_name"], how="left")

# format earliest timestamps as ISO strings (or None)
aggregated_logs_df["earliest_high_ts"] = aggregated_logs_df["earliest_high_ts"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
aggregated_logs_df["earliest_low_ts"]  = aggregated_logs_df["earliest_low_ts"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
aggregated_logs_df["earliest_high_ts"] = aggregated_logs_df["earliest_high_ts"].replace({pd.NaT: None, "NaT": None})
aggregated_logs_df["earliest_low_ts"]  = aggregated_logs_df["earliest_low_ts"].replace({pd.NaT: None, "NaT": None})

# 4) filter to entries with at least one anomaly and sort by combined anomaly count descending; limit to top 10
aggregated_logs_df["combined_anomalies"] = aggregated_logs_df["high_count"] + aggregated_logs_df["low_count"]
result_logs = (
    aggregated_logs_df[aggregated_logs_df["combined_anomalies"] > 0]
    .sort_values(by="combined_anomalies", ascending=False)
    .loc[:, ["cmdb_id", "log_name", "P95", "P05", "window_points", "high_count", "low_count",
             "earliest_high_ts", "earliest_low_ts", "max_value", "min_value"]]
    .head(10)
    .reset_index(drop=True)
)

# 5) up to 10 sample raw log rows from the incident window for these two cmdb_ids
sample_logs = window_logs_df.loc[:, ["timestamp", "cmdb_id", "log_name", "value"]].copy()
sample_logs["timestamp"] = sample_logs["timestamp"].dt.strftime("%Y-%m-%dT%H:%M:%S%z")
sample_logs = sample_logs.head(10).reset_index(drop=True)

# Keep intermediate dataframes in kernel variables:
# thresholds_logs_df, window_logs_df, window_with_thresholds_logs_df, aggregated_logs_df

# Display compact outputs: result_logs (top 10) and sample_logs (up to 10)
result_logs, sample_logs

```
Out[5]:
```
Summary (plain English):

- log.error_count for both checkoutservice and currencyservice is 0 across the window (P95=P05=0). All 20 sampled points equal 0 (earliest at 2024-01-16T09:32:00Z) — either truly no errors or a telemetry/normalization artifact.
- log.row_count shows one high and one low point per service within the window:
  - checkoutservice: P95 ≈ 105.3, P05 ≈ 8.7; window_points=20, high_count=1 (earliest high 2024-01-16T09:44:00Z, max=111), low_count=1 (earliest low 2024-01-16T09:56:00Z, min=3).
  - currencyservice: P95 ≈ 2475.5, P05 ≈ 390.25; window_points=20, high_count=1 (earliest high 2024-01-16T09:32:00Z, max=2542), low_count=1 (earliest low 2024-01-16T09:56:00Z, min=72).
- Concrete samples: at 09:32 currencyservice row_count=2542 (the high), checkoutservice row_count at 09:32 was 93 (not the high for checkout).
- Interpretation/action:
  - Verify whether error_count being all zeros is expected; if not, check log ingestion/normalization.
  - Investigate the currencyservice traffic spike at ~09:32 (row_count=2542) — it aligns with trace/metric anomalies seen earlier and may be a driver of downstream effects.
  - Check checkoutservice spike at ~09:44 and the low points at ~09:56 for both services to understand traffic drop/rebalance.

The original code execution output of IPython Kernel is also provided below for reference:

(           cmdb_id         log_name     P95     P05  window_points  high_count  low_count          earliest_high_ts           earliest_low_ts  max_value  min_value
0  checkoutservice  log.error_count     0.0    0.00             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000          0          0
1  currencyservice  log.error_count     0.0    0.00             20          20         20  2024-01-16T09:32:00+0000  2024-01-16T09:32:00+0000          0          0
2  checkoutservice    log.row_count   105.3    8.70             20           1          1  2024-01-16T09:44:00+0000  2024-01-16T09:56:00+0000        111          3
3  currencyservice    log.row_count  2475.5  390.25             20           1          1  2024-01-16T09:32:00+0000  2024-01-16T09:56:00+0000       2542         72,                   timestamp          cmdb_id         log_name  value
0  2024-01-16T09:32:00+0000  checkoutservice  log.error_count      0
1  2024-01-16T09:32:00+0000  checkoutservice    log.row_count     93
2  2024-01-16T09:32:00+0000  currencyservice  log.error_count      0
3  2024-01-16T09:32:00+0000  currencyservice    log.row_count   2542
4  2024-01-16T09:33:00+0000  checkoutservice  log.error_count      0
5  2024-01-16T09:33:00+0000  checkoutservice    log.row_count     96
6  2024-01-16T09:33:00+0000  currencyservice  log.error_count      0
7  2024-01-16T09:33:00+0000  currencyservice    log.row_count   2311
8  2024-01-16T09:34:00+0000  checkoutservice  log.error_count      0
9  2024-01-16T09:34:00+0000  checkoutservice    log.row_count    102)```
```